In [1]:
import os
os.chdir("../")

import sys
import torch
import json
from pathlib import Path

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
ckpt_path = "checkpoints/acevedo_sngp_resnet18/model.ckpt"

checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f"Checkpoint keys: {checkpoint.keys()}")

/home/wisc/maheswararao/code/lightning-hydra-template/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checkpoint keys: dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'datamodule_hparams_name', 'datamodule_hyper_parameters'])


In [4]:
def extract_state_dict(checkpoint, device):
    """Extract and remap state dict from Lightning checkpoint."""
    state_dict = checkpoint['state_dict']
    
    # Remove 'net.' prefix (Lightning module wrapper)
    remapped_state_dict = {}
    for key, value in state_dict.items():
        # Remove 'net.' prefix
        new_key = key.replace('net.', '', 1)
        remapped_state_dict[new_key] = value
    
    return remapped_state_dict

# Extract the clean state dict
clean_state_dict = extract_state_dict(checkpoint, device)
print(f"Remapped state dict keys (first 5):\n{list(clean_state_dict.keys())[:5]}")

Remapped state dict keys (first 5):
['backbone.0.weight_orig', 'backbone.0.weight_u', 'backbone.0.weight_v', 'backbone.1.weight', 'backbone.1.bias']


In [5]:
from src.models.sngp_gpt import SNGPClassifier

# Load original model
original_model = SNGPClassifier(num_classes = 8,
        arch = "resnet18",
        pretrained = False,
        rff_dim = 1024,
        length_scale = 1.0,
        ridge_penalty = 1e-3,
        cov_momentum = 0.999,
        mean_field = True,
        n_power_iterations_sn = 1)
original_model = original_model.to(device)
original_model.eval()

original_model.load_state_dict(clean_state_dict)

<All keys matched successfully>

In [6]:
test_input = torch.randn(2, 3, 224, 224).to(device)
print(f"✓ Created test input with shape: {test_input.shape}")

✓ Created test input with shape: torch.Size([2, 3, 224, 224])


In [7]:
with torch.no_grad():
    output = original_model(test_input)

mean_field_logits, raw_logits, pred_var = output
print(f"✓ Original model output shapes - mean_field_logits: {mean_field_logits.shape}, raw_logits: {raw_logits.shape}, pred_var: {pred_var.shape}")

✓ Original model output shapes - mean_field_logits: torch.Size([2, 8]), raw_logits: torch.Size([2, 8]), pred_var: torch.Size([2, 1])


In [8]:

mean_field_logits, raw_logits, pred_var

(tensor([[-0.3071, -0.1017,  0.1394, -0.4896, -0.3914, -0.7349,  0.0508,  1.2833],
         [-0.2640, -0.0710,  0.1083, -0.4214, -0.3772, -0.7127,  0.0999,  1.1171]],
        device='cuda:0'),
 tensor([[ -6.0476,  -2.0019,   2.7450,  -9.6401,  -7.7071, -14.4698,   1.0010,
           25.2682],
         [ -5.4654,  -1.4696,   2.2424,  -8.7243,  -7.8090, -14.7569,   2.0691,
           23.1298]], device='cuda:0'),
 tensor([[386.6950],
         [427.6956]], device='cuda:0'))

In [9]:
from transformers import AutoModel
from huggingface_hub import snapshot_download
from pathlib import Path

repo_id = "nirschl-lab/sngp-models"
variant_name = "acevedo_sngp_resnet18"

cache_dir = snapshot_download(
        repo_id=repo_id,
        allow_patterns=f"{variant_name}/*",
        cache_dir="./hf_models_cache",
        repo_type="model",
    )

model = AutoModel.from_pretrained(
        str(Path(cache_dir) / variant_name),
        trust_remote_code=True,
    )
model = model.to(device)
model.eval()

# test_input = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(test_input)
print(f"    Output shape: {output['logits'].shape}")

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 47527.52it/s]

    Output shape: torch.Size([2, 8])


In [10]:
output

{'mean_field_logits': tensor([[ 0.0164,  0.0008,  0.0019,  0.0503,  0.0362, -0.0166, -0.0119, -0.0527],
         [-0.0035,  0.0160, -0.0360, -0.0254,  0.0596, -0.0184,  0.0075, -0.0182]],
        device='cuda:0'),
 'logits': tensor([[ 0.0164,  0.0008,  0.0019,  0.0503,  0.0362, -0.0166, -0.0119, -0.0527],
         [-0.0035,  0.0160, -0.0360, -0.0254,  0.0596, -0.0184,  0.0075, -0.0182]],
        device='cuda:0'),
 'raw_logits': tensor([[ 0.5219,  0.0259,  0.0619,  1.6006,  1.1512, -0.5269, -0.3780, -1.6766],
         [-0.1084,  0.5008, -1.1269, -0.7958,  1.8667, -0.5779,  0.2365, -0.5696]],
        device='cuda:0'),
 'pred_var': tensor([[1010.6402],
         [ 980.9535]], device='cuda:0')}

In [13]:
mean_field_logits, raw_logits, pred_var

(tensor([[-0.3071, -0.1017,  0.1394, -0.4896, -0.3914, -0.7349,  0.0508,  1.2833],
         [-0.2640, -0.0710,  0.1083, -0.4214, -0.3772, -0.7127,  0.0999,  1.1171]],
        device='cuda:0'),
 tensor([[ -6.0476,  -2.0019,   2.7450,  -9.6401,  -7.7071, -14.4698,   1.0010,
           25.2682],
         [ -5.4654,  -1.4696,   2.2424,  -8.7243,  -7.8090, -14.7569,   2.0691,
           23.1298]], device='cuda:0'),
 tensor([[386.6950],
         [427.6956]], device='cuda:0'))